In [1]:
%pylab inline
import gc
import os
import glob
import numpy as np

import matplotlib
import matplotlib.pyplot as plt

import astropy.io.fits as pyfits
import astropy.table as astTable

import sim_analysis_utilities as utilities

Populating the interactive namespace from numpy and matplotlib


In [27]:
version='v2'
nomDir='wrkDir/s19/s19-1/anaCat_newS19Mask_fdeltacut/catalog_obs_reGaus_no_m/'
pzDir='wrkDir/s19/s19-1/anaCat_newS19Mask_fdeltacut/catalog_others/'
pozDir='wrkDir/database/s19-poz/origin/'
outDir='wrkDir/s19/s19-1/anaCat_newS19Mask_fdeltacut/catalog_tracts/'
if not os.path.isdir(outDir):
    os.mkdir(outDir)
nameDict={'dempz':'PDz.target_wide_s19a_%s.fits',
          'dnnz':'dNNz_target_wide_s19a_%s.fits',
          'mizuki':'target_wide_s19a_%s_all.fits'
         }

# nom and photoz

In [8]:
for fieldname in utilities.field_names: 
    outDir2=os.path.join(outDir,'%s_tracts' %fieldname)
    if not os.path.isdir(outDir2):
        os.mkdir(outDir2)
    print(fieldname)
    data=pyfits.getdata(os.path.join(nomDir,'%s_no_m.fits' %fieldname))
    inds=np.arange(len(data))
    zata=pyfits.getdata(os.path.join(pzDir,'%s_pz_2.fits' %fieldname))
    tractList=list(set(data['tract']))
    for tract in tractList:
        indsO=inds[data['tract']==tract]
        dataO=data[indsO]
        zataO=zata[indsO]
        pyfits.writeto(os.path.join(outDir2,'%s_no_m.fits'%tract),dataO)
        pyfits.writeto(os.path.join(outDir2,'%s_pz.fits'%tract),zataO)
        assert np.all(dataO['object_id']==zataO['object_id'])
        del dataO,zataO
        gc.collect()

XMM
GAMA09H
WIDE12H
GAMA15H
VVDS
HECTOMAP


# poz

In [23]:
info=astTable.Table.read('wrkDir/fieldTractInfoS19.csv')
for fieldname in utilities.field_names: 
    outDirT=os.path.join(outDir,'%s_tracts' %fieldname)
    fnamelist=glob.glob(os.path.join(outDir,'%s_tracts/*_pz.fits' %fieldname))
    infoU=info[info['field']==fieldname]
    assert len(fnamelist)==len(infoU)
    for tract in infoU['tract']:
        fname=os.path.join(outDirT,'%s_pz.fits' %(tract))
        assert os.path.isfile(fname)
        ids=astTable.Table.read(fname)['object_id',]
        ids['object_id'].colname='ID'
        
        for key in nameDict.keys():
            pzname=os.path.join(pozDir,key,nameDict[key] %(tract),hdu=1)
            assert os.path.isfile(pzname)
            fits=astTable.Table.read(pzname)
            join=astTab.join(left=fits,right=ids,keys='ID',join_type='inner')
            assert len(join)==len(ids)
            oname=os.path.join(outDirT,'%s_pz_pdf_%s.fits' %(tract,key))
            join.write(oname)
            del join,fits,pzname,oname
            gc.collect()

30 30


AssertionError: 

In [37]:
# The second HDU
info=astTable.Table.read('wrkDir/fieldTractInfoS19.csv')
for fieldname in utilities.field_names: 
    outDirT=os.path.join(outDir,'%s_tracts/' %fieldname)
    infoU=info[info['field']==fieldname]
    for tract in infoU['tract']:
        for key in nameDict.keys():
            pzname=os.path.join(pozDir,key,nameDict[key] %(tract))
            assert os.path.isfile(pzname)
            patade=pyfits.open(pzname)[2]
            outname=os.path.join(outDirT,'%s_pz_pdf_%s.fits' %(tract,key))
            assert os.path.isfile(outname)
            fits=pyfits.open(outname)
            fits.append(patade)
            fits.writeto(outname,overwrite=True)
            del fits,patade,outname,pzname
            gc.collect()